# VisDrone person detector: evaluation + video only (no training)

Use this when training is finished (or stopped) and you only need the final numbers and a demo video.
It loads `best.pt` from `My Drive/visdrone_person/runs/` and needs only the **validation** zips in `My Drive/visdrone_data/`
(`VisDrone2019-DET-val.zip`, and `VisDrone2019-VID-val.zip` for the video results).
Keep `TILE`, `OVERLAP` and `IMGSZ` the same as in training (defaults match). Needs a GPU runtime (Runtime > Change runtime type > T4).
Results are written to `My Drive/visdrone_person/results/`.

In [ ]:
!pip -q install "ultralytics>=8.3,<8.4" pycocotools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, math, random, shutil, subprocess, time, glob, io, contextlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU-memory fragmentation (must be set before torch starts)

# =====================  USER SETTINGS  =====================
QUICK_TEST      = False   # True = ~20-30 min pipeline check (tiny model, tiny data). Set False for the real run.
MODEL_SIZE      = "s"     # "n" | "s" | "m"  (s fits a free T4; m is better but ~2x slower)
TILE            = 512     # native-resolution crop size used for training AND inference (must match!)
OVERLAP         = 0.25    # tile overlap (must exceed typical person size so no person is cut in all tiles)
IMGSZ           = 768     # network input size. IMGSZ/TILE = 1.5x upscaling -> tiny people get more pixels
EPOCHS          = 30
BATCH           = 8       # 16 ran out of GPU memory in the box-matching step on a T4 (crowded tiles); 8 is safer
MAX_TRAIN_TILES = 16000   # cap on training tiles (time budget). None = use all
EMPTY_TILE_FRAC = 0.10    # share of person-free tiles kept as hard negatives
COPY_PASTE_PROB = 0.30    # prob. a train tile receives pasted small persons (box-level copy-paste)
NWD_WEIGHT      = 0.5     # box loss = (1-w)*CIoU + w*NWD
NWD_C           = 12.8    # NWD normalising constant in network-input pixels (value from the NWD paper)
USE_VID_TRAIN   = True    # add frames from VisDrone-VID train (real video motion blur); ~7.5 GB
VID_TRAIN_STRIDE = 8      # keep every 8th video frame (neighbouring frames are near-duplicates)
VID_EVAL_STRIDE  = 10     # evaluate on every 10th VID-val frame
EVAL_MAX_IMAGES  = None   # limit images for evaluation (None = all)
WORKERS          = 2      # data-loader processes; use 1 if Colab runs out of RAM (each worker holds ~1-2 GB)
REBUILD_TILES    = False  # False = reuse tiles already on local disk (e.g. after a kernel restart); True = rebuild from scratch

# ---- Google Drive (nothing is downloaded: the dataset is READ from your Drive) ----
DRIVE_FOLDER_NAME = "visdrone_person"   # results are written to My Drive/<this folder>/
DATA_FOLDER_NAME  = "visdrone_data"     # YOU upload the VisDrone zips into My Drive/<this folder>/ (see section 3)
# ===========================================================

if QUICK_TEST:
    MODEL_SIZE, EPOCHS, BATCH, MAX_TRAIN_TILES = "n", 2, 8, 600
    USE_VID_TRAIN, EVAL_MAX_IMAGES, VID_EVAL_STRIDE = False, 40, 60

WORK = Path(os.environ.get("WORK", "/content/visdrone_work"))   # fast local disk for tiles / unzipped data
WORK.mkdir(parents=True, exist_ok=True)

# ---- mount Google Drive (required: results are written straight to it) ----
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive")
except ImportError:                                    # not on Colab (local testing only)
    DRIVE_BASE = WORK / "fake_drive"
    print("!! google.colab not found -> using", DRIVE_BASE, "instead of Google Drive")
OUT     = DRIVE_BASE / DRIVE_FOLDER_NAME
RUNS    = OUT / "runs"            # Ultralytics training runs (weights, curves) - written every epoch
RESULTS = OUT / "results"         # metrics, predictions, demo video, detections JSON
WEIGHTS = OUT / "weights"         # final best.pt copy
DATA_DIR = DRIVE_BASE / DATA_FOLDER_NAME   # input: the VisDrone zips (or unzipped folders) you uploaded
for d in (RUNS, RESULTS, WEIGHTS, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

import gc
def ram(tag=""):
    """Print system RAM use. Call it anywhere to see where memory goes (Colab free = ~12.7 GB)."""
    try:
        import psutil
        vm, rss = psutil.virtual_memory(), psutil.Process().memory_info().rss
        kids = sum(p.memory_info().rss for p in psutil.Process().children(recursive=True))
        print(f"[RAM {tag}] system {vm.used/2**30:.1f}/{vm.total/2**30:.1f} GB | this notebook {rss/2**30:.1f} GB + workers {kids/2**30:.1f} GB")
    except Exception:
        pass

SEED = 0
random.seed(SEED); np.random.seed(SEED)
(RESULTS / "config_eval.json").write_text(json.dumps(
    {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, type(None)))}, indent=1))
print("local work dir:", WORK, "\nGoogle Drive output:", OUT)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable Runtime > Change runtime type > T4 GPU)")
except ImportError:
    pass

Mounted at /content/drive
local work dir: /content/visdrone_work 
Google Drive output: /content/drive/MyDrive/visdrone_person
GPU: Tesla T4


In [ ]:
# Reads the dataset from YOUR Google Drive folder (DATA_DIR). Nothing is downloaded from the internet.
RAW = WORK / "raw"; RAW.mkdir(exist_ok=True)

SPLITS = {   # key -> name prefix of the zip / folder you uploaded
    "DET-train": "VisDrone2019-DET-train",
    "DET-val":   "VisDrone2019-DET-val",
    "VID-train": "VisDrone2019-VID-train",
    "VID-val":   "VisDrone2019-VID-val",
}

def _copy_low_ram(src, dst, chunk=64 * 2**20):
    """Copy a big Drive file to local disk in 64 MB chunks, dropping the OS file cache as we go
    (reading a multi-GB zip straight from the Drive mount can fill Colab's RAM)."""
    with open(src, "rb") as fi, open(dst, "wb") as fo:
        while True:
            b = fi.read(chunk)
            if not b:
                break
            fo.write(b)
            for f in (fi, fo):
                try:
                    f.flush(); os.fsync(f.fileno()) if f is fo else None
                    os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                except Exception:
                    pass

def _unzip(zpath, dest):
    subprocess.run(["unzip", "-q", "-o", str(zpath), "-d", str(dest)], check=True)

def _find_root(dest):
    a = next(Path(dest).rglob("annotations"), None)
    return a.parent if a else None

def fetch(name, required):
    """Copy/unzip one split from Drive to fast local disk. Returns the folder that holds 'annotations/', or None."""
    dest = RAW / name
    if dest.exists() and _find_root(dest):
        return _find_root(dest)
    prefix = SPLITS[name].lower()
    found = sorted(p for p in DATA_DIR.iterdir() if p.name.lower().startswith(prefix))
    try:
        if not found:
            raise FileNotFoundError(f"nothing named '{SPLITS[name]}*' in {DATA_DIR}")
        src = next((p for p in found if p.is_dir()), None) or next((p for p in found if p.suffix.lower() == ".zip"), None)
        if src is None:
            raise FileNotFoundError(f"'{found[0].name}' is neither a .zip nor a folder")
        dest.mkdir(parents=True, exist_ok=True)
        print(f"{name}: reading {src.name} from Drive ...")
        if src.is_dir():
            shutil.copytree(src, dest, dirs_exist_ok=True)
        else:
            local_zip = RAW / f"{name}.zip"
            _copy_low_ram(src, local_zip)      # Drive -> local disk first, then unzip from local disk
            _unzip(local_zip, dest)
            local_zip.unlink(missing_ok=True)
        root = _find_root(dest)
        assert root, f"no 'annotations' folder found inside {src.name} (is the zip complete?)"
        gc.collect(); ram(f"after {name}")
        return root
    except Exception as e:
        shutil.rmtree(dest, ignore_errors=True)
        msg = f"{name}: {e}"
        if required:
            have = [p.name for p in DATA_DIR.iterdir()] if DATA_DIR.exists() else []
            raise SystemExit(f"\n!! REQUIRED dataset missing -> {msg}\nFiles currently in {DATA_DIR}: {have}\n"
                             f"Upload the VisDrone zips there (see section 3 of this notebook), then re-run this cell.")
        print(f"(optional) {msg} -> skipping")
        return None

DET_TRAIN = None            # not needed: evaluation only
DET_VAL   = fetch("DET-val", True)
VID_VAL   = fetch("VID-val", False)
VID_TRAIN = None
print("DET-train:", DET_TRAIN, "\nDET-val:  ", DET_VAL, "\nVID-train:", VID_TRAIN, "\nVID-val:  ", VID_VAL)

DET-val: reading VisDrone2019-DET-val.zip from Drive ...
[RAM after DET-val] system 1.6/12.7 GB | this notebook 0.6 GB + workers 0.0 GB
VID-val: reading VisDrone2019-VID-val.zip from Drive ...
[RAM after VID-val] system 1.7/12.7 GB | this notebook 0.6 GB + workers 0.0 GB
DET-train: None 
DET-val:   /content/visdrone_work/raw/DET-val/VisDrone2019-DET-val 
VID-train: None 
VID-val:   /content/visdrone_work/raw/VID-val/VisDrone2019-VID-val


In [ ]:
# VisDrone categories: 0 ignored-region, 1 pedestrian, 2 people, 3 bicycle, 4 car, ... -> we merge 1+2 into one class "person"
PERSON_CATS = (1, 2)

def _arr(x):
    return np.array(x, np.float32).reshape(-1, 4)

def parse_det_file(path):
    """DET row: x,y,w,h,score,category,truncation,occlusion  ->  (person boxes xyxy, ignore regions xyxy)"""
    boxes, ign = [], []
    p = Path(path)
    if p.exists():
        for line in p.read_text().splitlines():
            f = line.strip().split(",")
            if len(f) < 6:
                continue
            x, y, w, h, score, cat = (int(float(v)) for v in f[:6])
            if w <= 0 or h <= 0:
                continue
            b = (x, y, x + w, y + h)
            if cat == 0 or (score == 0 and cat in PERSON_CATS):
                ign.append(b)
            elif cat in PERSON_CATS:
                boxes.append(b)
    return _arr(boxes), _arr(ign)

def parse_vid_file(path):
    """VID row: frame,id,x,y,w,h,score,category,truncation,occlusion -> {frame: (boxes, ignore)}"""
    per = defaultdict(lambda: ([], []))
    for line in Path(path).read_text().splitlines():
        f = line.strip().split(",")
        if len(f) < 8:
            continue
        fr, _, x, y, w, h, score, cat = (int(float(v)) for v in f[:8])
        if w <= 0 or h <= 0:
            continue
        b = (x, y, x + w, y + h)
        if cat == 0 or (score == 0 and cat in PERSON_CATS):
            per[fr][1].append(b)
        elif cat in PERSON_CATS:
            per[fr][0].append(b)
    return {k: (_arr(v[0]), _arr(v[1])) for k, v in per.items()}

def det_records(root):
    root = Path(root); out = []
    for p in sorted((root / "images").glob("*.jpg")):
        b, i = parse_det_file(root / "annotations" / (p.stem + ".txt"))
        out.append(dict(uid=f"det_{p.stem}", path=str(p), boxes=b, ignore=i))
    return out

def vid_records(root, stride=1):
    root = Path(root); out = []
    for seq in sorted(d for d in (root / "sequences").iterdir() if d.is_dir()):
        ann = parse_vid_file(root / "annotations" / f"{seq.name}.txt")
        frames = sorted(seq.glob("*.jpg"))
        for p in frames[::stride]:
            b, i = ann.get(int(p.stem), (_arr([]), _arr([])))
            out.append(dict(uid=f"vid_{seq.name}_{p.stem}", path=str(p), boxes=b, ignore=i))
    return out

def summarize(records, name):
    n = sum(len(r["boxes"]) for r in records)
    sides = np.concatenate([np.sqrt((r["boxes"][:, 2]-r["boxes"][:, 0]) * (r["boxes"][:, 3]-r["boxes"][:, 1])) for r in records if len(r["boxes"])] or [np.zeros(0)])
    print(f"{name}: {len(records)} images, {n} person boxes, "
          f"{(sides < 32).mean()*100:.0f}% COCO-small (<32px side), median side {np.median(sides):.0f}px")

In [ ]:
def tile_origins(W, H, tile, overlap):
    step = max(1, int(tile * (1 - overlap)))
    def axis(L):
        if L <= tile:
            return [0]
        xs = list(range(0, L - tile + 1, step))
        if xs[-1] != L - tile:
            xs.append(L - tile)
        return xs
    return [(x, y) for y in axis(H) for x in axis(W)]

def boxes_in_tile(boxes, x0, y0, tw, th, min_vis=0.5):
    """Boxes (xyxy, image coords) -> boxes clipped to the tile; drops boxes <min_vis visible or <2px."""
    if len(boxes) == 0:
        return np.zeros((0, 4), np.float32)
    b = boxes.copy()
    b[:, [0, 2]] -= x0; b[:, [1, 3]] -= y0
    area0 = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    c = b.copy()
    c[:, [0, 2]] = c[:, [0, 2]].clip(0, tw); c[:, [1, 3]] = c[:, [1, 3]].clip(0, th)
    cw, ch = c[:, 2] - c[:, 0], c[:, 3] - c[:, 1]
    vis = (cw.clip(0) * ch.clip(0)) / np.maximum(area0, 1e-6)
    return c[(vis >= min_vis) & (cw >= 2) & (ch >= 2)]

def paint_ignore(img, ign):
    """Grey-out VisDrone 'ignored regions' so un-annotated people there are not learned as background."""
    for x1, y1, x2, y2 in ign.astype(int):
        img[max(y1, 0):max(y2, 0), max(x1, 0):max(x2, 0)] = 114

def plan_tiles(records, tile, overlap, min_vis=0.5):
    plan = []
    for ri, r in enumerate(tqdm(records, desc="planning tiles")):
        W, H = Image.open(r["path"]).size
        for x0, y0 in tile_origins(W, H, tile, overlap):
            tw, th = min(tile, W - x0), min(tile, H - y0)
            plan.append((ri, x0, y0, tw, th, len(boxes_in_tile(r["boxes"], x0, y0, tw, th, min_vis))))
    return plan

def sample_plan(plan, max_tiles, empty_frac, rng):
    pos = [p for p in plan if p[5] > 0]; emp = [p for p in plan if p[5] == 0]
    n_pos = len(pos) if max_tiles is None else min(len(pos), int(max_tiles * (1 - empty_frac)))
    n_emp = min(len(emp), int(n_pos * empty_frac / max(1 - empty_frac, 1e-6)))
    return rng.sample(pos, n_pos) + rng.sample(emp, n_emp)

# ---------- box-level copy-paste of SMALL persons (Ultralytics' copy_paste needs masks, VisDrone has only boxes) ----------
def _overlaps(box, others):
    """True if axis-aligned `box` (x1,y1,x2,y2) intersects any box in `others`."""
    if len(others) == 0:
        return False
    o = np.asarray(others, np.float32).reshape(-1, 4)
    ix = np.minimum(box[2], o[:, 2]) - np.maximum(box[0], o[:, 0])
    iy = np.minimum(box[3], o[:, 3]) - np.maximum(box[1], o[:, 1])
    return bool(((ix > 0) & (iy > 0)).any())

def add_donors(pool, crop, lab, rng, pool_size=3000, small_side=48, per_tile=2):
    th, tw = crop.shape[:2]; added = 0
    for i in rng.sample(range(len(lab)), len(lab)):
        x1, y1, x2, y2 = lab[i]; bw, bh = x2 - x1, y2 - y1
        if max(bw, bh) > small_side or added >= per_tile:
            continue
        m = max(3, int(0.3 * max(bw, bh)))
        reg = np.array([x1 - m, y1 - m, x2 + m, y2 + m])
        if reg[0] < 0 or reg[1] < 0 or reg[2] > tw or reg[3] > th:
            continue
        if _overlaps(reg, np.delete(lab, i, axis=0)):
            continue
        patch = crop[int(reg[1]):int(reg[3]), int(reg[0]):int(reg[2])].copy()
        if patch.size == 0:
            continue
        item = dict(patch=patch, m=m)
        if len(pool) < pool_size:
            pool.append(item)
        else:
            pool[rng.randrange(pool_size)] = item
        added += 1

def paste_donors(crop, lab, pool, rng, max_paste=5):
    th, tw = crop.shape[:2]; lab = lab.copy(); n = 0
    for _ in range(rng.randint(1, max_paste)):
        d = rng.choice(pool); patch, m = d["patch"], d["m"]; ph, pw = patch.shape[:2]
        if pw >= tw or ph >= th:
            continue
        for _try in range(10):
            px, py = rng.randint(0, tw - pw), rng.randint(0, th - ph)
            region = np.array([px, py, px + pw, py + ph], np.float32)
            if _overlaps(region, lab):
                continue
            alpha = np.zeros((ph, pw), np.float32)
            k = max(m // 2, 1)
            alpha[k:ph - k, k:pw - k] = 1.0
            ks = 2 * (m // 2) + 1
            alpha = cv2.GaussianBlur(alpha, (ks, ks), 0)[..., None]
            roi = crop[py:py + ph, px:px + pw].astype(np.float32)
            crop[py:py + ph, px:px + pw] = (alpha * patch + (1 - alpha) * roi).astype(np.uint8)
            lab = np.vstack([lab, [px + m, py + m, px + pw - m, py + ph - m]]).astype(np.float32)
            n += 1
            break
    return crop, lab, n

def write_tiles(records, selected, img_dir, lbl_dir, paint=True, copy_paste_prob=0.0, rng=None):
    rng = rng or random.Random(0)
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    by_rec = defaultdict(list)
    for s in selected:
        by_rec[s[0]].append(s)
    order = list(by_rec); rng.shuffle(order)
    pool, n_tiles, n_boxes, n_pasted = [], 0, 0, 0
    for ri in tqdm(order, desc="writing tiles"):
        r = records[ri]
        img = cv2.imread(r["path"])
        if img is None:
            continue
        if paint:
            paint_ignore(img, r["ignore"])
        for _, x0, y0, tw, th, _n in by_rec[ri]:
            crop = img[y0:y0 + th, x0:x0 + tw].copy()
            lab = boxes_in_tile(r["boxes"], x0, y0, tw, th)
            if copy_paste_prob > 0:
                if len(lab):
                    add_donors(pool, crop, lab, rng)
                if pool and rng.random() < copy_paste_prob:
                    crop, lab, k = paste_donors(crop, lab, pool, rng); n_pasted += k
            name = f"{r['uid']}_{x0}_{y0}"
            cv2.imwrite(str(img_dir / f"{name}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(lbl_dir / f"{name}.txt", "w") as f:
                for x1, y1, x2, y2 in lab:
                    f.write(f"0 {(x1+x2)/2/tw:.6f} {(y1+y2)/2/th:.6f} {(x2-x1)/tw:.6f} {(y2-y1)/th:.6f}\n")
            n_tiles += 1; n_boxes += len(lab)
    print(f"wrote {n_tiles} tiles, {n_boxes} boxes ({n_pasted} pasted) -> {img_dir}")

In [ ]:
det_val = det_records(DET_VAL)
vid_val = vid_records(VID_VAL, VID_EVAL_STRIDE) if VID_VAL else []
summarize(det_val, "DET val")
if vid_val: summarize(vid_val, "VID val (subsampled)")
BEST = next(iter(sorted(RUNS.glob("person_p2_nwd_*/weights/best.pt"))), None)
assert BEST, f"no best.pt found under {RUNS} - was the training run saved to Drive?"
print("using weights:", BEST)

DET val: 548 images, 13969 person boxes, 89% COCO-small (<32px side), median side 17px
VID val (subsampled): 288 images, 5096 person boxes, 25% COCO-small (<32px side), median side 47px
using weights: /content/drive/MyDrive/visdrone_person/runs/person_p2_nwd_s/weights/best.pt


In [ ]:
def nms_np(boxes, scores, thr):
    order = scores.argsort()[::-1]; keep = []
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    while len(order):
        i = order[0]; keep.append(i)
        if len(order) == 1:
            break
        r = order[1:]
        w = (np.minimum(boxes[i, 2], boxes[r, 2]) - np.maximum(boxes[i, 0], boxes[r, 0])).clip(0)
        h = (np.minimum(boxes[i, 3], boxes[r, 3]) - np.maximum(boxes[i, 1], boxes[r, 1])).clip(0)
        inter = w * h
        order = r[inter / (area[i] + area[r] - inter + 1e-9) <= thr]
    return np.array(keep, dtype=int)

def nms(boxes, scores, thr):
    try:
        import torch, torchvision
        return torchvision.ops.nms(torch.from_numpy(boxes).float(), torch.from_numpy(scores).float(), thr).numpy()
    except ImportError:
        return nms_np(boxes, scores, thr)

class SlicedDetector:
    """Tiled (SAHI-style) person detector. Tiles use the SAME size/overlap as training.
    mode="sliced": tiles (+ optional full-frame pass for large people) -> merged with NMS.
    mode="full"  : a single full-frame pass at `full_imgsz` (baseline for comparison)."""
    def __init__(self, weights=None, model=None, tile=TILE, overlap=OVERLAP, imgsz=IMGSZ, conf=0.001,
                 tile_iou=0.6, merge_iou=0.6, mode="sliced", full_pass=True, full_imgsz=1280, full_min_side=28,
                 tile_batch=16, max_det=500, edge_margin=2):
        if model is None:
            from ultralytics import YOLO
            model = YOLO(str(weights))
        self.model, self.tile, self.overlap, self.imgsz, self.conf = model, tile, overlap, imgsz, conf
        self.tile_iou, self.merge_iou, self.mode = tile_iou, merge_iou, mode
        self.full_pass, self.full_imgsz, self.full_min_side = full_pass, full_imgsz, full_min_side
        self.tile_batch, self.max_det, self.edge_margin = tile_batch, max_det, edge_margin
        try:
            import torch; self.half = torch.cuda.is_available()
        except ImportError:
            self.half = False

    def _infer(self, imgs, imgsz):
        res = self.model.predict(imgs, imgsz=imgsz, conf=self.conf, iou=self.tile_iou, half=self.half,
                                 max_det=300, verbose=False)
        out = []
        for r in res:
            b = r.boxes
            out.append(np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]], 1).astype(np.float32)
                       if len(b) else np.zeros((0, 5), np.float32))
        return out

    def _finalize(self, d):
        if len(d) == 0:
            return np.zeros((0, 5), np.float32)
        d = d[nms(d[:, :4], d[:, 4], self.merge_iou)]
        return d[np.argsort(-d[:, 4])][: self.max_det]

    def __call__(self, img):
        H, W = img.shape[:2]
        if self.mode == "full":
            return self._finalize(self._infer([img], self.full_imgsz)[0])
        dets, origins, m = [], tile_origins(W, H, self.tile, self.overlap), self.edge_margin
        for i in range(0, len(origins), self.tile_batch):
            chunk = origins[i:i + self.tile_batch]
            crops = [img[y:y + self.tile, x:x + self.tile] for x, y in chunk]
            for (x, y), crop, d in zip(chunk, crops, self._infer(crops, self.imgsz)):
                if len(d) == 0:
                    continue
                th, tw = crop.shape[:2]
                keep = np.ones(len(d), bool)           # drop boxes cut by an INTERIOR tile border (a neighbour tile sees them whole)
                if x > 0:       keep &= d[:, 0] > m
                if y > 0:       keep &= d[:, 1] > m
                if x + tw < W:  keep &= d[:, 2] < tw - m
                if y + th < H:  keep &= d[:, 3] < th - m
                d = d[keep].copy()
                d[:, [0, 2]] += x; d[:, [1, 3]] += y
                dets.append(d)
        if self.full_pass:                              # large people that no single tile contains
            d = self._infer([img], self.full_imgsz)[0]
            if len(d):
                dets.append(d[np.sqrt((d[:, 2] - d[:, 0]) * (d[:, 3] - d[:, 1])) >= self.full_min_side])
        return self._finalize(np.concatenate(dets) if dets else np.zeros((0, 5), np.float32))

In [ ]:
def blur_score(img):
    """Variance of the Laplacian: LOW = blurry. Computed at a fixed width so scores are comparable across frames."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    s = 960.0 / max(g.shape)
    if s < 1:
        g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

def in_ignore(dets, ign, thr=0.5):
    """VisDrone protocol: a detection mostly (>=50% of its area) inside an ignored region is neither TP nor FP."""
    if len(ign) == 0 or len(dets) == 0:
        return np.zeros(len(dets), bool)
    d, g = dets[:, None, :4], ign[None, :, :]
    wh = (np.minimum(d[..., 2:], g[..., 2:]) - np.maximum(d[..., :2], g[..., :2])).clip(0)
    inter = wh[..., 0] * wh[..., 1]
    area = ((dets[:, 2] - dets[:, 0]) * (dets[:, 3] - dets[:, 1]))[:, None]
    return ((inter / np.maximum(area, 1e-9)) >= thr).any(1)

def run_detection(detector, records, max_images=None, desc="detect"):
    recs = records[:max_images] if max_images else records
    preds, blur, t0 = [], [], time.time()
    for i, r in enumerate(tqdm(recs, desc=desc)):
        img = cv2.imread(r["path"])
        blur.append(blur_score(img))
        d = detector(img)
        d = d[~in_ignore(d, r["ignore"])]
        for x1, y1, x2, y2, c in d:
            preds.append(dict(image_id=i + 1, category_id=1, bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)], score=float(c)))
    return recs, preds, np.array(blur), (time.time() - t0) / max(len(recs), 1)

def build_gt(records):
    images, anns, aid = [], [], 1
    for i, r in enumerate(records):
        W, H = Image.open(r["path"]).size
        images.append(dict(id=i + 1, width=W, height=H, file_name=Path(r["path"]).name))
        for x1, y1, x2, y2 in r["boxes"]:
            w, h = float(x2 - x1), float(y2 - y1)
            anns.append(dict(id=aid, image_id=i + 1, category_id=1, bbox=[float(x1), float(y1), w, h], area=w * h, iscrowd=0)); aid += 1
    return dict(images=images, annotations=anns, categories=[dict(id=1, name="person")])

def coco_eval(records, preds, img_ids=None, max_dets=100):
    """Official COCO bbox evaluation (IoU .50:.95). small = area < 32^2 px, medium < 96^2, large >= 96^2."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    keys = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]
    gt = COCO()
    gt.dataset = build_gt(records)
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    if img_ids is not None:
        ids = set(int(i) for i in img_ids); preds = [p for p in preds if p["image_id"] in ids]
    if not preds:
        return {k: 0.0 for k in keys}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(preds)
        E = COCOeval(gt, dt, "bbox")
        E.params.maxDets = [1, 10, max_dets]
        if img_ids is not None:
            E.params.imgIds = sorted(ids)
        E.evaluate(); E.accumulate(); E.summarize()
    return {k: float(v) for k, v in zip(keys, E.stats[:6])}

def report(records, preds, blur, title):
    """Headline table + sharp-vs-blurry breakdown (blurriest 25% of frames vs the rest)."""
    rows = {}
    rows["all frames  (maxDets=100, standard COCO)"] = coco_eval(records, preds)
    rows["all frames  (maxDets=500, crowded scenes)"] = coco_eval(records, preds, max_dets=500)
    ids = np.arange(1, len(records) + 1); thr = np.percentile(blur, 25)
    rows["blurriest 25% of frames"] = coco_eval(records, preds, ids[blur <= thr], 500)
    rows["sharper 75% of frames"]   = coco_eval(records, preds, ids[blur > thr], 500)
    print(f"\n=== {title} ===  ({len(records)} images)")
    print(f"{'':44s}" + "".join(f"{k:>10s}" for k in ["AP", "AP50", "AP_small", "AP_med", "AP_large"]))
    for name, m in rows.items():
        print(f"{name:44s}" + "".join(f"{m[k]*100:10.1f}" for k in ["AP", "AP50", "AP_small", "AP_medium", "AP_large"]))
    return rows

In [ ]:
BEST_W = BEST if "BEST" in globals() else next(iter(sorted(RUNS.glob("person_p2_nwd_*/weights/best.pt"))))
det = SlicedDetector(BEST_W)

# --- image-detection val set (DET) ---
recs, preds, blur, sec = run_detection(det, det_val, EVAL_MAX_IMAGES, "DET val | sliced")
res_sliced = report(recs, preds, blur, "VisDrone-DET val | sliced inference")
print(f"~{sec:.2f} s / frame")

# --- ablation: plain full-frame inference at 1280 (shows what slicing buys) ---
gc.collect(); ram("after sliced eval")
det.mode = "full"
recs2, preds2, blur2, sec2 = run_detection(det, det_val, EVAL_MAX_IMAGES, "DET val | full-frame 1280")
res_full = report(recs2, preds2, blur2, "VisDrone-DET val | full-frame @1280 (baseline)")
det.mode = "sliced"

# --- video val set (VID frames) : this is the number that matters for video detection ---
if vid_val:
    recs3, preds3, blur3, sec3 = run_detection(det, vid_val, EVAL_MAX_IMAGES, "VID val | sliced")
    res_vid = report(recs3, preds3, blur3, "VisDrone-VID val frames | sliced inference")
    BLUR_THR = float(np.percentile(blur3, 25))
else:
    BLUR_THR = float(np.percentile(blur, 25))
print("blur threshold (25th percentile of val frames):", round(BLUR_THR, 1))

# --- save everything to Google Drive ---
ALL = {"DET_sliced": res_sliced, "DET_fullframe_1280": res_full}
if vid_val:
    ALL["VID_sliced"] = res_vid
(RESULTS / "metrics.json").write_text(json.dumps(ALL, indent=1))
with open(RESULTS / "metrics.csv", "w") as f:
    f.write("dataset,subset,AP,AP50,AP75,AP_small,AP_medium,AP_large\n")
    for ds, rows in ALL.items():
        for sub, m in rows.items():
            f.write(f"{ds},{sub.strip()}," + ",".join(f"{m[k]:.4f}" for k in ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]) + "\n")
(RESULTS / "predictions_DET_val_sliced.json").write_text(json.dumps(preds))
shutil.copy(BEST_W, WEIGHTS / "best.pt")
print("saved metrics.json / metrics.csv / predictions / best.pt to", OUT)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


DET val | sliced:   0%|          | 0/548 [00:00<?, ?it/s]


=== VisDrone-DET val | sliced inference ===  (548 images)
                                                    AP      AP50  AP_small    AP_med  AP_large
all frames  (maxDets=100, standard COCO)          33.6      70.6      31.1      49.8      71.6
all frames  (maxDets=500, crowded scenes)       -100.0      73.7      32.5      50.6      71.6
blurriest 25% of frames                         -100.0      78.1      33.4      52.0      70.2
sharper 75% of frames                           -100.0      72.6      32.3      50.0      75.5
~0.14 s / frame
[RAM after sliced eval] system 3.0/12.7 GB | this notebook 2.3 GB + workers 0.0 GB


DET val | full-frame 1280:   0%|          | 0/548 [00:00<?, ?it/s]


=== VisDrone-DET val | full-frame @1280 (baseline) ===  (548 images)
                                                    AP      AP50  AP_small    AP_med  AP_large
all frames  (maxDets=100, standard COCO)          27.7      62.2      24.5      48.1      72.5
all frames  (maxDets=500, crowded scenes)       -100.0      64.7      25.4      48.9      72.4
blurriest 25% of frames                         -100.0      71.3      27.8      51.2      71.9
sharper 75% of frames                           -100.0      62.9      24.8      47.8      76.3


VID val | sliced:   0%|          | 0/288 [00:00<?, ?it/s]


=== VisDrone-VID val frames | sliced inference ===  (288 images)
                                                    AP      AP50  AP_small    AP_med  AP_large
all frames  (maxDets=100, standard COCO)          25.5      62.4       7.0      31.1      40.8
all frames  (maxDets=500, crowded scenes)       -100.0      63.0       7.2      31.8      41.3
blurriest 25% of frames                         -100.0      53.1       3.1      24.3      40.3
sharper 75% of frames                           -100.0      67.5       8.7      35.5      41.9
blur threshold (25th percentile of val frames): 554.7
saved metrics.json / metrics.csv / predictions / best.pt to /content/drive/MyDrive/visdrone_person


In [ ]:
CONF_SHARP, CONF_BLUR = 0.30, 0.20   # blurred frames produce weaker scores -> use a lower confidence cut-off there

def process_video(src, dst, detector, max_frames=None, blur_thr=None, save_json=None):
    """Per-frame person detection on a video file (no tracking). Returns the list of per-frame detections."""
    blur_thr = BLUR_THR if blur_thr is None else blur_thr
    cap = cv2.VideoCapture(str(src))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
    raw = str(WORK / "_raw_video.mp4")   # write locally, then encode straight into Drive
    out = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    log, t0, k = [], time.time(), 0
    pbar = tqdm(total=min(n, max_frames) if (n and max_frames) else (max_frames or n), desc="video")
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames and k >= max_frames):
            break
        blurry = blur_score(frame) < blur_thr
        d = detector(frame); d = d[d[:, 4] >= (CONF_BLUR if blurry else CONF_SHARP)]
        for x1, y1, x2, y2, c in d:
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 255) if blurry else (0, 255, 0), 1)
        cv2.putText(frame, f"persons: {len(d)}  {'BLURRY' if blurry else 'sharp'}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        out.write(frame)
        log.append(dict(frame=k, blurry=bool(blurry), boxes=[[float(v) for v in row] for row in d]))
        k += 1; pbar.update(1)
    cap.release(); out.release(); pbar.close()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-vcodec", "libx264", "-pix_fmt", "yuv420p", str(dst)], check=False)
    if save_json:
        Path(save_json).write_text(json.dumps(log))
    print(f"{k} frames in {time.time()-t0:.0f}s ({(time.time()-t0)/max(k,1):.2f} s/frame) -> {dst}")
    return log

# ---- demo: turn the first VID-val sequence into a short clip, run it, and show it ----
if VID_VAL:
    seq = sorted(d for d in (Path(VID_VAL) / "sequences").iterdir() if d.is_dir())[0]
    frames = sorted(seq.glob("*.jpg"))[:150]
    h, w = cv2.imread(str(frames[0])).shape[:2]
    demo_in = WORK / "demo_in.mp4"
    vw = cv2.VideoWriter(str(demo_in), cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
    for f in frames: vw.write(cv2.imread(str(f)))
    vw.release()
    process_video(demo_in, RESULTS / "demo_out.mp4", det, save_json=RESULTS / "demo_detections.json")
    from IPython.display import Video, display
    display(Video(str(RESULTS / "demo_out.mp4"), embed=True, width=800))

# ---- your own video: upload it in the Colab file panel (or point to a file in your Drive), then run e.g. ----
# process_video("/content/my_drone_clip.mp4", RESULTS / "my_drone_clip_out.mp4", det, save_json=RESULTS / "my_drone_clip.json")

video:   0%|          | 0/150 [00:00<?, ?it/s]

150 frames in 39s (0.26 s/frame) -> /content/drive/MyDrive/visdrone_person/results/demo_out.mp4
